In [57]:
import re
from bs4 import BeautifulSoup

# Article 1's HTML, as a standalone string, so we can test in isolation
article_1_html = """
<div class="eli-subdivision" id="art_1">
   <p id="d1e1384-1-1" class="oj-ti-art">Article 1</p>
   <div class="eli-title" id="art_1.tit_1">
      <p class="oj-sti-art">Subject-matter and objectives</p>
   </div>
   <div id="001.001">
      <p class="oj-normal">1.   This Regulation lays down rules relating to the protection of natural persons with regard to the processing of personal data and rules relating to the free movement of personal data.</p>
   </div>
   <div id="001.002">
      <p class="oj-normal">2.   This Regulation protects fundamental rights and freedoms of natural persons and in particular their right to the protection of personal data.</p>
   </div>
   <div id="001.003">
      <p class="oj-normal">3.   The free movement of personal data within the Union shall be neither restricted nor prohibited for reasons connected with the protection of natural persons with regard to the processing of personal data.</p>
   </div>
</div>
"""

soup = BeautifulSoup(article_1_html, "lxml-xml")

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

# 1. Get Article Number

article_div = soup.find("div", class_= 'eli-subdivision')
number_tag = article_div.find("p", class_='oj-ti-art')
number_text = clean_text(number_tag.get_text())
article_number = re.search(r"Article\s+(\S+)", number_text).group(1)

# 2. Get heading

title_div = article_div.find("div", class_='eli-title')
title_tag = title_div.find('p', class_= 'oj-sti-art')
heading = clean_text(title_tag.get_text())

# 3. Get paragraphs
paragraphs = []
paragraph_divs = article_div.find_all("div", recursive=False)
for div in paragraph_divs:
    paragraph_tag = div.find('p', class_='oj-normal')
    if paragraph_tag is None:
        continue

    raw_text = clean_text(paragraph_tag.get_text())
    match = re.match(r"^(\d+)\.\s*(.*)$", raw_text)
    label, text = match.group(1), match.group(2)
    paragraphs.append({'label': label, 'text': text})

# Assemble everything

article_record = {
    "document": "32016R0679",  # GDPR's CELEX -- known in advance, hardcoded for this test
    "article_id": article_div.get("id"),
    "article_number": article_number,
    "heading": heading,
    "chapter_id": "cpt_I",           # from context -- Article 1 sits directly in Chapter I
    "chapter_heading": "General provisions",
    "section_id": None,              # Chapter I has no sections
    "section_heading": None,
    "paragraphs": paragraphs,
    "word_count": len(" ".join(p['text'] for p in paragraphs).split()),
}

import json
print(json.dumps(article_record, indent=2))

{
  "document": "32016R0679",
  "article_id": "art_1",
  "article_number": "1",
  "heading": "Subject-matter and objectives",
  "chapter_id": "cpt_I",
  "chapter_heading": "General provisions",
  "section_id": null,
  "section_heading": null,
  "paragraphs": [
    {
      "label": "1",
      "text": "This Regulation lays down rules relating to the protection of natural persons with regard to the processing of personal data and rules relating to the free movement of personal data."
    },
    {
      "label": "2",
      "text": "This Regulation protects fundamental rights and freedoms of natural persons and in particular their right to the protection of personal data."
    },
    {
      "label": "3",
      "text": "The free movement of personal data within the Union shall be neither restricted nor prohibited for reasons connected with the protection of natural persons with regard to the processing of personal data."
    }
  ],
  "word_count": 83
}
